# BCI Motor Imagery : Quick Overview
- **What we are doing**: Looking at brain waves when someone imagines moving their left or right hand.
- **Why we filter it (8–30 Hz)**: Brain signals are noisy. This filter cuts out the junk and keeps only the brain rhythms (mu and beta waves) that matter for hand movement.

### Step 1: Load Pure Motor Imagery Data & Apply Band-Pass Filter
* **Files Used**: `S001R04.edf`, `S001R08.edf`, `S001R12.edf` (Left vs. Right Hand Motor Imagery).
* **Signal Processing**: Applied an **8–30 Hz Band-Pass Filter** to isolate Mu (8–12 Hz) and Beta (13–30 Hz) frequency bands critical for motor cortex activity.

In [ ]:
from pathlib import Path
import mne

# Define your base file directory
DATA_DIR = Path.home() / "Desktop" / "files"

# Pure Motor Imagery (Left vs. Right Hand) - Runs 4, 8, 12
run_files = [
    DATA_DIR / "S001" / "S001R04.edf",
    DATA_DIR / "S001" / "S001R08.edf",
    DATA_DIR / "S001" / "S001R12.edf",
]

# Load and combine the file(s) automatically
raw_list = [mne.io.read_raw_edf(f, preload=True, verbose=False) for f in run_files]
if len(raw_list) > 1:
    raw = mne.concatenate_raws(raw_list)
else:
    raw = raw_list[0]

# Apply your 8–30 Hz band-pass filter to clean the noise
raw.filter(l_freq=8.0, h_freq=30.0, fir_design='firwin', verbose=False)

print(f"✅ Successfully loaded {len(run_files)} file(s) and filtered the data!")

### Step 2: Epoching and Common Spatial Patterns (CSP)
* **Epoch Extraction**: Sliced data from $0.0\text{s}$ to $3.0\text{s}$ post-cue.
* **Spatial Filtering**: Utilized **CSP** (`n_components=4`) to project multi-channel EEG data into a low-dimensional space that maximizes variance between left and right hand imagery classes.

In [ ]:
# 3. Plot the filtered EEG signals to get your graph for the paper
fig = raw.plot(duration=5, start=10, n_channels=8, scalings="auto", title="Filtered EEG Motor Imagery Signals")

### Step 3: LDA Classification & Cross-Validation
* **Classifier**: Linear Discriminant Analysis (LDA).
* **Validation**: 5-Fold Stratified Cross-Validation.
* **Baseline Accuracy**: ~64.4% (reflecting the challenge of decoding pure, silent motor imagery without physical muscle movement).

In [ ]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import ShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline

# 1. Find the event markers (T1 = left fist, T2 = right fist)
events, event_id = mne.events_from_annotations(raw, verbose=False)
selected_events = {'T1': event_id['T1'], 'T2': event_id['T2']}

# 2. Chop data into 3-second trials for left vs. right imagination
epochs = mne.Epochs(raw, events, event_id=selected_events, tmin=0.0, tmax=3.0, baseline=None, preload=True, verbose=False)

X = epochs.get_data()  # The brain wave trials
y = epochs.events[:, -1]  # The labels (Left vs. Right)

# 3. Build Machine Learning Pipeline: CSP (Spatial filtering) + LDA (Classifier)
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)
lda = LinearDiscriminantAnalysis()
clf = Pipeline([('CSP', csp), ('LDA', lda)])

# 4. Test the model using cross-validation
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')

print(f"\n--- RESULTS FOR YOUR PAPER ---")
print(f"Mean Classification Accuracy: {scores.mean() * 100:.2f}%")